In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from cvxopt import matrix, solvers

from sklearn import datasets
from sklearn import model_selection
from sklearn.datasets import make_circles
import plotly.graph_objects as go
from numba import njit

import neal

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import Dwave_Classic_SVM_CVXOPT_Gaussian as CG
import Dwave_Quantum_SVM_Gaussian_neal_QAC as QG

import Processing as PC

import os
import json
import time
import pandas as pd

In [2]:
SAVE_DIR_METRIC = r"/home/csb/SVM/Support-Vector-Machine-2/Quantum_SVM_DATA/other_data"    # CSV 저장 폴더
SAVE_DIR_ALPHA  = r"/home/csb/SVM/Support-Vector-Machine-2/Quantum_SVM_DATA/alpha_data"  # NPZ 저장 폴더

os.makedirs(SAVE_DIR_METRIC, exist_ok=True)
os.makedirs(SAVE_DIR_ALPHA,  exist_ok=True)

In [ ]:
# =========================
# Alpha NPZ 저장 경로 생성
# =========================
def build_alpha_path(seed: int, run_id: int, i: int, gamma: float, B: int, K: int, xi: int, C: float) -> str:
    # seed별 폴더로 분리 (원치 않으면 이 2줄 제거하고 바로 SAVE_DIR_ALPHA에 저장하면 됨)
    seed_dir = os.path.join(SAVE_DIR_ALPHA, f"D'Wave_seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    # 파일명 구성
    # (gamma는 소수점 포함 → 파일명 안전하게 포맷)
    return os.path.join(
        seed_dir,
        f"alpha_seed{seed}_run{run_id:03d}_idx{i:02d}_g{gamma:.2f}_B{B}_K{K}_xi{xi}_C{int(C)}.npz"
    )

In [ ]:
# =========================
# Main experiment
#   - 기존 코드 유지
#   - α 저장만 추가 (C_alpha, Q_alpha(top-k), Q_energy)
# =========================
def run_single_experiment(N_train, X_train, y_train, X_test, y_test, run_id, split_id, nested_C, lambda_penalty, seed):
    rows = []

    B_list = [5]*30
    K_list = [3]*30
    xi_list = [1]*30
    gamma_list = [g/10 for g in range(30)]

    for i in range(len(gamma_list)):
        B = B_list[i]; K = K_list[i]; xi = xi_list[i]; gamma = gamma_list[i]

        # ---- C 계산 (기존 유지)
        C = 0
        for k in range(K):
            C += B**k

        # =======================
        # Quantum SVM (top-k)
        # =======================
        K_train_train_q, Q = QG.Q_metric(N_train, X_train, y_train, B, K, xi, gamma, nested_C, lambda_penalty)
        
        label_name = f"CSB_QSVM_B=seed={seed},{B},K={K},xi={xi},gamma={gamma}"
        sol_q = QG.Quantum_Solver(Q_upper=Q, label_name=label_name)

        print("gamma =", gamma)
        print("len(sol_q) =", len(sol_q))
        print("top_k =", min(len(sol_q), 20))
        print("energies =", sol_q.record.energy)
        print("num_occurrences =", sol_q.record.num_occurrences)

        top_k = min(len(sol_q), 20)

        # --- alpha/energy 저장 리스트 (추가)
        Q_alpha_list = []
        Q_energy_list = []

        # --- metrics lists
        Q_acc_test_list = []
        Q_auroc_test_list = []
        Q_auprc_test_list = []

        Q_acc_train_list = []
        Q_auroc_train_list = []
        Q_auprc_train_list = []

        Q_gap_acc_list = []
        Q_gap_auroc_list = []
        Q_gap_auprc_list = []

        Q_loss_train_list = []
        Q_loss_test_list = []

        Q_J_w_list = []
        Q_J_xi_list = []

        for n_th in range(top_k):
            x_opt, energy = QG.Solution(sol_q, top_k, n_th)
            alpha_q = QG.alpha_value(N_train, x_opt, B, K, nested_C)

            # ---- α/energy 저장 (추가)
            Q_alpha_list.append(alpha_q.astype(np.float64, copy=True))
            Q_energy_list.append(float(energy))

            Q_acc_train, Q_auroc_train, Q_auprc_train, Q_scores_train = QG.evaluate_train(
                y_train, alpha_q, K_train_train_q, C
            )
            Q_acc_test, Q_auroc_test, Q_auprc_test = QG.evaluate_test(
                y_test,
                QG.Test_evaluation(X_train, X_test, y_train, alpha_q, K_train_train_q, gamma, C)
            )

            Q_gap_acc, Q_gap_auroc, Q_gap_auprc = QG.Evaluate_Overfitting(
                Q_acc_train, Q_acc_test, Q_auroc_train, Q_auroc_test, Q_auprc_train, Q_auprc_test
            )

            Q_loss_train, Q_loss_test = QG.Hinge_Loss(
                X_train, X_test, y_train, y_test, alpha_q, K_train_train_q, Q_scores_train, gamma, C
            )

            Q_J_w, Q_J_xi = QG.Primal(alpha_q, K_train_train_q, y_train, C)
            Q_J_w_list.append(Q_J_w)
            Q_J_xi_list.append(Q_J_xi)

            Q_acc_test_list.append(Q_acc_test)
            Q_auroc_test_list.append(Q_auroc_test)
            Q_auprc_test_list.append(Q_auprc_test)

            Q_acc_train_list.append(Q_acc_train)
            Q_auroc_train_list.append(Q_auroc_train)
            Q_auprc_train_list.append(Q_auprc_train)

            Q_gap_acc_list.append(Q_gap_acc)
            Q_gap_auroc_list.append(Q_gap_auroc)
            Q_gap_auprc_list.append(Q_gap_auprc)

            Q_loss_train_list.append(Q_loss_train)
            Q_loss_test_list.append(Q_loss_test)

        # =======================
        # Quantum summary stats
        # =======================
        if len(Q_J_w_list) > 0:
            Q_acc_test_max = float(np.max(Q_acc_test_list))
            Q_auroc_test_max = float(np.max(Q_auroc_test_list))
            Q_auprc_test_max = float(np.max(Q_auprc_test_list))

            Q_acc_test_min = float(np.min(Q_acc_test_list))
            Q_auroc_test_min = float(np.min(Q_auroc_test_list))
            Q_auprc_test_min = float(np.min(Q_auprc_test_list))

            Q_acc_test_mean = float(np.mean(Q_acc_test_list))
            Q_auroc_test_mean = float(np.mean(Q_auroc_test_list))
            Q_auprc_test_mean = float(np.mean(Q_auprc_test_list))

            Q_acc_train_max = float(np.max(Q_acc_train_list))
            Q_auroc_train_max = float(np.max(Q_auroc_train_list))
            Q_auprc_train_max = float(np.max(Q_auprc_train_list))

            Q_acc_train_min = float(np.min(Q_acc_train_list))
            Q_auroc_train_min = float(np.min(Q_auroc_train_list))
            Q_auprc_train_min = float(np.min(Q_auprc_train_list))

            Q_acc_train_mean = float(np.mean(Q_acc_train_list))
            Q_auroc_train_mean = float(np.mean(Q_auroc_train_list))
            Q_auprc_train_mean = float(np.mean(Q_auprc_train_list))

            Q_gap_acc_max = float(np.max(Q_gap_acc_list))
            Q_gap_auroc_max = float(np.max(Q_gap_auroc_list))
            Q_gap_auprc_max = float(np.max(Q_gap_auprc_list))

            Q_gap_acc_min = float(np.min(Q_gap_acc_list))
            Q_gap_auroc_min = float(np.min(Q_gap_auroc_list))
            Q_gap_auprc_min = float(np.min(Q_gap_auprc_list))

            Q_gap_acc_mean = float(np.mean(Q_gap_acc_list))
            Q_gap_auroc_mean = float(np.mean(Q_gap_auroc_list))
            Q_gap_auprc_mean = float(np.mean(Q_gap_auprc_list))

            Q_hinge_gap_mean = float(np.mean(Q_loss_test_list) - np.mean(Q_loss_train_list))

            Q_J_w_max  = float(np.max(Q_J_w_list))
            Q_J_w_min  = float(np.min(Q_J_w_list))
            Q_J_w_mean = float(np.mean(Q_J_w_list))

            Q_J_xi_max  = float(np.max(Q_J_xi_list))
            Q_J_xi_min  = float(np.min(Q_J_xi_list))
            Q_J_xi_mean = float(np.mean(Q_J_xi_list))
        else:
            Q_acc_test_max = Q_auroc_test_max = Q_auprc_test_max = np.nan
            Q_acc_test_min = Q_auroc_test_min = Q_auprc_test_min = np.nan
            Q_acc_test_mean = Q_auroc_test_mean = Q_auprc_test_mean = np.nan
            Q_acc_train_max = Q_auroc_train_max = Q_auprc_train_max = np.nan
            Q_acc_train_min = Q_auroc_train_min = Q_auprc_train_min = np.nan
            Q_acc_train_mean = Q_auroc_train_mean = Q_auprc_train_mean = np.nan
            Q_gap_acc_max = Q_gap_auroc_max = Q_gap_auprc_max = np.nan
            Q_hinge_gap_mean = np.nan

            Q_J_w_max = Q_J_w_min = np.nan
            Q_J_xi_max = Q_J_xi_min = np.nan

            Q_J_w_mean = np.nan
            Q_J_xi_mean = np.nan

        # =======================
        # α 파일 저장 (추가)
        # =======================
        if len(Q_alpha_list) > 0:
            Q_alpha_arr = np.stack(Q_alpha_list, axis=0)   # (top_k, N_train)
            Q_energy_arr = np.array(Q_energy_list, dtype=np.float64)
        else:
            Q_alpha_arr = np.empty((0, N_train), dtype=np.float64)
            Q_energy_arr = np.empty((0,), dtype=np.float64)

        alpha_path = build_alpha_path(seed, run_id, i, gamma, B, K, xi, C)

        np.savez_compressed(
            alpha_path,
            Q_alpha=Q_alpha_arr,
            Q_energy=Q_energy_arr
        )

        # =======================
        # CSV row (기존 + alpha_path만 추가)
        # =======================
        rows.append({
            "seed": seed,
            "run": run_id,
            "i": i,
            "B": B, "K": K, "xi": xi, "gamma": gamma, "C": C,

            "Q_acc_train_list" : Q_acc_train_list, "Q_auroc_train_list" : Q_auroc_train_list,
            "Q_auprc_train_list" : Q_auprc_train_list,

            "Q_acc_test_list" : Q_acc_test_list, "Q_auroc_test_list" : Q_auroc_test_list,
            "Q_auprc_test_list" : Q_auprc_test_list,

            "Q_gap_acc_list" : Q_gap_acc_list,
            "Q_gap_auroc_list" : Q_gap_auroc_list,
            "Q_gap_auprc_list" : Q_gap_auprc_list,

            "Q_loss_train_list" : Q_loss_train_list,
            "Q_loss_test_list" : Q_loss_test_list,

            "Q_J_w_list" : Q_J_w_list,
            "Q_J_xi_list" : Q_J_xi_list,

            "Q_acc_test_max": Q_acc_test_max,
            "Q_auroc_test_max": Q_auroc_test_max,
            "Q_auprc_test_max": Q_auprc_test_max,

            "Q_acc_test_min": Q_acc_test_min,
            "Q_auroc_test_min": Q_auroc_test_min,
            "Q_auprc_test_min": Q_auprc_test_min,

            "Q_acc_test_mean": Q_acc_test_mean,
            "Q_auroc_test_mean": Q_auroc_test_mean,
            "Q_auprc_test_mean": Q_auprc_test_mean,

            "Q_acc_train_max": Q_acc_train_max,
            "Q_auroc_train_max": Q_auroc_train_max,
            "Q_auprc_train_max": Q_auprc_train_max,

            "Q_acc_train_min": Q_acc_train_min,
            "Q_auroc_train_min": Q_auroc_train_min,
            "Q_auprc_train_min": Q_auprc_train_min,

            "Q_acc_train_mean": Q_acc_train_mean,
            "Q_auroc_train_mean": Q_auroc_train_mean,
            "Q_auprc_train_mean": Q_auprc_train_mean,

            "Q_gap_acc_max": Q_gap_acc_max,
            "Q_gap_auroc_max": Q_gap_auroc_max,
            "Q_gap_auprc_max": Q_gap_auprc_max,

            "Q_gap_acc_min": Q_gap_acc_min,
            "Q_gap_auroc_min": Q_gap_auroc_min,
            "Q_gap_auprc_min": Q_gap_auprc_min,

            "Q_gap_acc_mean": Q_gap_acc_mean,
            "Q_gap_auroc_mean": Q_gap_auroc_mean,
            "Q_gap_auprc_mean": Q_gap_auprc_mean,

            "Q_hinge_gap_mean": Q_hinge_gap_mean,

            "Q_J_w_max": Q_J_w_max,
            "Q_J_w_min": Q_J_w_min,
            "Q_J_w_mean": Q_J_w_mean,

            "Q_J_xi_max": Q_J_xi_max,
            "Q_J_xi_min": Q_J_xi_min,
            "Q_J_xi_mean": Q_J_xi_mean,

            # 추가: α 파일 경로
            "alpha_npz_path": alpha_path,
            "Q_topk": int(Q_alpha_arr.shape[0]),
        })

    return rows

In [ ]:
N_RUNS = 1

for seed in range(43, 44):
    all_rows = []
    n_train = 200
    raw = 5
    col = int(n_train / raw)
    nested_C = 3
    lambda_penalty = 1

    X, Y = make_circles(n_samples=500, noise=0.1, random_state=seed)
    X_train, y_train, X_test, y_test = PC.Processing(X, Y, n_train, raw, col)
    kernal_size = len(X_train[0])

    for run_id in range(1, N_RUNS + 1):
        print(f"[SEED {seed}] [RUN {run_id}/{N_RUNS}]")
        for split_id in range(raw):
            all_rows.extend(run_single_experiment(
                kernal_size, 
                X_train[split_id],
                y_train[split_id],
                X_test,
                y_test,
                run_id,
                split_id,
                nested_C,
                lambda_penalty,
                seed)
                )
            
    df = pd.DataFrame(all_rows)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"D'Wave_SVM_SVM_seed{seed}_runs{N_RUNS}_{timestamp}.csv"
    save_path = os.path.join(SAVE_DIR_METRIC, filename)

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"Saved results to:\n{save_path}")
    print(f"Alpha NPZ saved under:\n{SAVE_DIR_ALPHA}")

[SEED 43] [RUN 1/1]
gamma = 0.0
len(sol_q) = 20
top_k = 20
energies = [-9018. -9018. -8802. -8802. -8802. -8802. -8802. -8800. -8568. -8568.
 -8568. -8568. -8568. -8568. -8352. -8352. -8352. -8352. -8352. -8352.]
num_occurrences = [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
gamma = 0.1
len(sol_q) = 20
top_k = 20
energies = [-5772.51513059 -5650.28442299 -5644.56919769 -5572.29653119
 -5562.43262733 -5553.12689288 -5515.40330586 -5508.69285151
 -5506.53028257 -5504.94554975 -5455.30147452 -5451.13993534
 -5438.2695763  -5434.87732086 -5431.84104294 -5426.83717608
 -5413.10167175 -5393.79694154 -5388.0230723  -5386.04177777]
num_occurrences = [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
gamma = 0.2
len(sol_q) = 20
top_k = 20
energies = [-3781.63431184 -3750.58690823 -3664.31918177 -3639.59929125
 -3607.74090989 -3595.87879672 -3549.76672922 -3535.21462122
 -3525.18737226 -3519.90974716 -3512.20139952 -3482.83917722
 -3472.33230257 -3462.71331132 -3459.51218764 -3436.1863819
 -3413.99523147 -3